# Okanka Mini — 01: tokenizer check + data preparation (CPU notebook)

Run this **once**, on a **CPU** session (Settings: Accelerator = None, Internet = On). CPU sessions do **not**
use your weekly GPU quota.

What it does:
1. Verifies the committed Turkish tokenizer (`tokenizer/`) loads and reports its compression.
2. Downloads FineWeb-2 `tur_Latn` + Turkish Wikipedia over HTTPS, writes nanochat parquet shards
   (`text` column, row groups of 1024 docs), and uploads them to the private HF dataset repo
   `<HF_USERNAME>/okanka-mini-data` (deleting local copies to stay under the disk limit).
3. Builds the Turkish SFT jsonl (Aya + OASST2 + Wikipedia template QA) + identity data and uploads it to the same repo.

Needed Kaggle secrets: `HF_TOKEN` (HF **write** token), optional `HF_USERNAME`, optional `GITHUB_TOKEN` (if the code repo is private).

Time: ~1.5-3 h (mostly parquet processing + upload). If the session dies, just re-run: finished shards are
already on the hub; set `START_SHARD` to the number of shards already uploaded and `SKIP_DOCS` accordingly — or simply delete the dataset repo and start over.

In [ ]:
# ======================= OKANKA MINI CONFIG (edit if needed) =======================
REPO_URL   = "https://github.com/alisimseksembol/okanka-mini.git"   # GitHub repo with this code (public, or add Kaggle secret GITHUB_TOKEN)
BRANCH     = "main"
DEPTH      = 16          # model depth (d16 ~ 235M "scaling" params). Use 12 for a faster/cheaper run.
SEQ_LEN    = 1024        # context length used for pretraining on T4
DEVICE_BS  = 4 if DEPTH >= 14 else 8   # per-GPU micro batch (power of 2; halve it if CUDA OOM)
DATA_RATIO = 12          # tokens : scaling-params ratio (nanochat compute-optimal ~10-12; Chinchilla 20)
# Hugging Face repos (created automatically, private):
#   <HF_USERNAME>/okanka-mini-data         (dataset: pretraining shards + SFT jsonl)
#   <HF_USERNAME>/okanka-mini-checkpoints  (model: resumable training checkpoints)
#   <HF_USERNAME>/okanka-mini              (model: final public export, notebook 04)
# ===================================================================================

In [ ]:
import os, sys, subprocess, shutil
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name, default=None):
    try:
        return secrets.get_secret(name)
    except Exception:
        return default

os.environ["HF_TOKEN"] = secret("HF_TOKEN") or ""
assert os.environ["HF_TOKEN"], "Add a Kaggle secret named HF_TOKEN (HF write token): Add-ons -> Secrets"
HF_USERNAME = secret("HF_USERNAME")
if not HF_USERNAME:
    from huggingface_hub import whoami
    HF_USERNAME = whoami(token=os.environ["HF_TOKEN"])["name"]
DATA_REPO = f"{HF_USERNAME}/okanka-mini-data"
CKPT_REPO = f"{HF_USERNAME}/okanka-mini-checkpoints"
FINAL_REPO = f"{HF_USERNAME}/okanka-mini"
print("HF user:", HF_USERNAME)

# --- code ---
gh = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{gh}@") if gh else REPO_URL
CODE = "/kaggle/working/okanka-mini"
if not os.path.exists(CODE):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, CODE], check=True)
else:
    subprocess.run(["git", "-C", CODE, "pull"], check=False)
os.chdir(CODE)
sys.path.insert(0, CODE)

# --- python deps (torch comes with the Kaggle image) ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken>=0.11.0", "rustbpe>=0.1.0", "pyarrow>=21.0.0",
                "filelock", "psutil", "huggingface_hub>=0.34.0", "wandb"], check=True)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
major, minor = (int(x) for x in torch.__version__.split(".")[:2])
if (major, minor) < (2, 6):
    # nanochat needs a recent torch (SDPA enable_gqa, F.rms_norm, compile fixes). cu126 wheels still support T4 (sm75) and P100 (sm60).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1", "--index-url", "https://download.pytorch.org/whl/cu126"], check=True)
    print("Upgraded torch -> restart the kernel (Run > Restart) and run all cells again."); raise SystemExit

# --- nanochat runtime env ---
os.environ["NANOCHAT_BASE_DIR"] = "/tmp/nanochat"           # big ephemeral disk; persistent state lives on the HF hub
os.environ["NANOCHAT_DISABLE_FA3"] = "1"                    # FA3 is Hopper/Ampere-only; T4 uses PyTorch SDPA
os.environ["OMP_NUM_THREADS"] = "1"
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    if cap >= (8, 0):
        os.environ["NANOCHAT_DTYPE"] = "bfloat16"
    elif cap >= (7, 0):
        os.environ["NANOCHAT_DTYPE"] = "float16"            # T4/V100: fp16 tensor cores + GradScaler (nanochat built-in)
    else:
        os.environ["NANOCHAT_DTYPE"] = "float32"            # P100: no tensor cores, triton unsupported -> no compile
        os.environ["TORCHDYNAMO_DISABLE"] = "1"
NGPU = max(1, torch.cuda.device_count())
print("NANOCHAT_DTYPE =", os.environ.get("NANOCHAT_DTYPE"), "| GPUs:", NGPU)
os.makedirs(os.environ["NANOCHAT_BASE_DIR"], exist_ok=True)

def run(cmd):
    """Run a shell command, streaming its output into the notebook; raise on failure."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"command failed ({rc}): {cmd}")

In [ ]:
# 1) tokenizer sanity check (the tokenizer was trained from scratch and committed to the repo)
from nanochat.tokenizer import get_tokenizer
tok = get_tokenizer()
s = "Çağrı, İstanbul'daki ağaçların gölgesinde ılık bir öğle yemeği yedi."
ids = tok.encode(s)
print(tok.get_vocab_size(), len(s), "chars ->", len(ids), "tokens:", [tok.decode([i]) for i in ids])
assert tok.decode(ids) == s

In [ ]:
# 2) How much text do we need?  tokens = DATA_RATIO * scaling_params(DEPTH); chars ≈ tokens * chars_per_token / (1 - crop_loss)
import math, json
SCALING_PARAMS = {12: 110_100_912, 14: 164_233_804, 16: 234_881_792, 18: 324_404_172, 20: 435_160_240}  # transformer matrices + lm_head, vocab 32768
CHARS_PER_TOKEN = 4.55     # measured on our shards incl. full docs (tokenizer/compression_report.md: 4.85 on FineWeb-2 test)
CROP_LOSS = 0.54           # measured: best-fit packing at T=1024 keeps only ~46% of tokens (long docs get cropped)
tokens_needed = DATA_RATIO * SCALING_PARAMS[DEPTH]
TARGET_CHARS = int(tokens_needed * CHARS_PER_TOKEN / (1 - CROP_LOSS) * 1.1)
FW2_FILES = list(range(max(1, math.ceil(TARGET_CHARS / 14e9))))   # each FineWeb-2 tur_Latn train file holds ~15B chars (30 files total)
print(f"tokens needed ≈ {tokens_needed/1e9:.2f}B -> writing ≈ {TARGET_CHARS/1e9:.1f}B chars from FineWeb-2 files {FW2_FILES}")
START_SHARD = 0

In [ ]:
# 3) build + upload the shards (raw downloads go to /tmp, shards are deleted locally after upload)
run(f"python -m okanka.prepare_pretrain_data --target-chars {TARGET_CHARS} --fw2-files {' '.join(map(str, FW2_FILES))} "
    f"--download-dir /tmp/raw --delete-raw --upload-repo {DATA_REPO} --delete-after-upload --start-shard-idx {START_SHARD}")

In [ ]:
# 4) SFT + identity data -> data repo (sft/ folder)
run("python -m okanka.make_identity --n 260 --out data/identity_tr.jsonl")
run("python -m okanka.prepare_sft_data --raw-dir /tmp/raw/sft --wiki-dir /tmp/raw --out-dir data/sft --wiki-qa 12000")
from huggingface_hub import HfApi
api = HfApi(token=os.environ["HF_TOKEN"])
for f in ["data/sft/sft_tr_train.jsonl", "data/sft/sft_tr_val.jsonl", "data/identity_tr.jsonl"]:
    api.upload_file(path_or_fileobj=f, path_in_repo="sft/" + os.path.basename(f), repo_id=DATA_REPO, repo_type="dataset")
print([f for f in api.list_repo_files(DATA_REPO, repo_type="dataset")][:10], "...")